# PatchQuestions

An executable resource for budgeted API-contract disambiguation. Run all cells in a fresh CPU kernel. No network, model, GPU or pip install is needed for these experiments. Reference runtime is CPython 3.12.14. This notebook materializes nine source modules in a new writable subdirectory and preserves the embedded reference matrices separately. No learned-model session is performed.


In [ ]:
import base64, contextlib, gzip, io, json, os, runpy, sys, tempfile
from pathlib import Path
work = Path(tempfile.mkdtemp(prefix="patchquestions-", dir=Path.cwd())).resolve()
os.chdir(work)
sys.path.insert(0, str(work))
print("Runtime:", sys.version)
print("Output directory:", work.name)


In [ ]:
Path('benchmark.py').write_text('"""Offline, executable API-contract recovery benchmark. Python >=3.11."""\nimport base64, binascii, csv, fnmatch, hashlib, io, itertools, json, math, random, re\nimport statistics, textwrap\nfrom collections import Counter, defaultdict\nfrom functools import lru_cache\nfrom urllib.parse import parse_qsl, quote, quote_plus\nfrom pathlib import Path\n\ndef product(*axes):\n    return list(itertools.product(*axes))\n\ndef outcome(f, config, x):\n    try:\n        return json.dumps(["return", f(config,x)], ensure_ascii=True, sort_keys=True, allow_nan=False)\n    except (ValueError, csv.Error, UnicodeError, binascii.Error) as exc:\n        return json.dumps(["raise",type(exc).__name__])\n\ndef families():\n    # Each row contains candidate configurations, one smoke input, finite universe, implementation.\n    return {\n        "url_quote": (product([0,1],["","/","/:","@","/@"],["strict","replace"]), "cat",\n            [a+b for a,b in product(["hi","a b","/x","@",":","\\ud800"],[""," b","/x","@y",":z","é","+"])],\n            lambda c,x:(quote,quote_plus)[c[0]](x,safe=c[1],errors=c[2])),\n        "query_pairs": (product([False,True],[False,True],["&",";"],["latin1","utf8"]), "a=1",\n            [a+b for a,b in product(["a=1","b=","bare","x=%C3%A9","x=%FF",""],["","&q",";b=",";x=%FF","&b=2","+%2B"])],\n            lambda c,x:parse_qsl(x,keep_blank_values=c[0],strict_parsing=c[1],separator=c[2],encoding=c[3])),\n        "csv_fields": (product([",",";","\\t"],[False,True],[False,True],[None,"\\\\"]), "x",\n            [a+d+b for a,d,b in product([\'x\',\'"a"\',\'"a""b"\',\'a\\\\\'],[\',\',\';\',\'\\t\'],[\' y\',\'"x\',\'z\',\'\\\\q\'])],\n            lambda c,x:list(csv.reader([x],delimiter=c[0],skipinitialspace=c[1],strict=c[2],escapechar=c[3]))),\n        "wrap": (product([6,9],[False,True],[False,True],[False,True]), "hi",\n            [a+s+b for a,s,b in product(["ab","abcdefgh","a-bcd","abcde"],[" ","  ","-","\\t"],["z","efghijk","abc-def"])],\n            lambda c,x:textwrap.wrap(x,width=c[0],break_long_words=c[1],break_on_hyphens=c[2],drop_whitespace=c[3])),\n        "json_wire": (product([False,True],[False,True],[False,True],[False,True]), "1",\n            [json.dumps({a:v,"a":w},ensure_ascii=False) for a,v,w in product(["z","é","B"],["é",[1,2],0],["x",["雪"],{}])] + [\'[]\',\'{}\',\'"é"\',\'[1, 2]\'],\n            lambda c,x:json.dumps(json.loads(x),ensure_ascii=c[0],sort_keys=c[1],separators=((\',\' if c[2] else \', \'),(\':\' if c[3] else \': \')))),\n        "regex_search": (product([0,re.I],[0,re.M],[0,re.S],[0,re.A]), "a|a",\n            [p+\'|\'+s for p,s in product(["a","^b","a.b",r"\\w+",r"\\d+","[a-z]+","^a.b"],["A","a\\nb","x\\nb","é","９","aXb","K","!x"])],\n            lambda c,x:[m.span() for m in re.finditer(x.split(\'|\')[0],x.split(\'|\')[1],flags=int(c[0]|c[1]|c[2]|c[3]))]),\n        "base64_read": (product([None,b\'-_\',b\'._\'],[False,True]), "YQ==",\n            [a+b for a,b in product(["AA","-_","._","+/","a!","//","ab","YWJj"],["==","/w==","\\n==","!","_w==","", "==x"])],\n            lambda c,x:base64.b64decode(x,altchars=c[0],validate=c[1]).hex()),\n        "hex_format": (product([":","-"," "],[-3,-2,-1,1,2,3],[False,True]), "",\n            [bytes((v+i)%256 for i in range(k)).hex() for k,v in product(range(1,12),[0,10,120,230])],\n            lambda c,x:bytes.fromhex(x).hex(c[0],c[1]).upper() if c[2] else bytes.fromhex(x).hex(c[0],c[1]))\n    }\n\ndef signature(row, cols):\n    return tuple(row[j] for j in cols)\n\ndef evaluate(name, spec, seed, replacements=(), transfer_inputs=None):\n    configs, smoke, inputs, f=spec\n    inputs=sorted(set(inputs)-{smoke})\n    random.Random(seed).shuffle(inputs)\n    probes,hold=inputs[:min(16,len(inputs)//2)],inputs[min(16,len(inputs)//2):]\n    if replacements:\n        assert len(set(replacements)) == len(replacements)\n        assert not set(replacements).intersection(inputs+[smoke])\n        probes=probes[:-len(replacements)]+list(replacements)\n    if transfer_inputs is not None:\n        assert not set(transfer_inputs).intersection(probes+[smoke])\n        hold=list(transfer_inputs)\n    smokevals=[outcome(f,c,smoke) for c in configs]\n    query=[tuple(outcome(f,c,x) for x in probes) for c in configs]\n    hidden=[tuple(outcome(f,c,x) for x in hold) for c in configs]\n    @lru_cache(None)\n    def partition(ids,j):\n        bins=defaultdict(list)\n        for i in ids: bins[query[i][j]].append(i)\n        return tuple(tuple(v) for v in bins.values())\n    @lru_cache(None)\n    def tree(ids,depth,frugal):\n        # Maximize leaf count, optionally breaking ties by total path length over targets.\n        best=(1,0,None)\n        if depth==0: return best\n        for j in range(len(probes)):\n            parts=partition(ids,j)\n            if len(parts)<2: continue\n            children=[tree(p,depth-1,frugal) for p in parts]\n            leaves=sum(child[0] for child in children)\n            cost=len(ids)+sum(child[1] for child in children)\n            if leaves>best[0] or (frugal and leaves==best[0] and cost<best[1]):\n                best=(leaves,cost,j)\n        return best\n    methods=[\'random\',\'random_split\',\'diversity\',\'entropy\',\'lookahead\',\'frugal\']\n    rows=[]\n    for target in range(len(configs)):\n        initial=tuple(i for i in range(len(configs)) if smokevals[i]==smokevals[target])\n\n        @lru_cache(None)\n        def score_state(ids, available, depth, method):\n            if depth:\n                if method==\'random\':\n                    choices=available\n                else:\n                    informative=tuple(j for j in available if len(partition(ids,j))>1)\n                    if method==\'random_split\':\n                        choices=informative\n                    elif method in (\'lookahead\',\'frugal\'):\n                        _,_,j=tree(ids,depth,method==\'frugal\')\n                        choices=() if j is None else (j,)\n                    else:\n                        def utility(j):\n                            sizes=[len(p) for p in partition(ids,j)]\n                            if method==\'diversity\':\n                                return len(sizes)\n                            return -sum(n*math.log2(n) for n in sizes)\n                        choices=(max(informative,key=utility),) if informative else ()\n                if choices:\n                    results=[]\n                    for j in choices:\n                        survivors=next(p for p in partition(ids,j) if target in p)\n                        remaining=tuple(k for k in available if k!=j)\n                        results.append(score_state(survivors,remaining,depth-1,method))\n                    held,probe,count,used=map(statistics.fmean,zip(*results))\n                    return held,probe,count,used+1\n            return (sum(hidden[i]==hidden[target] for i in ids)/len(ids),\n                    sum(query[i]==query[target] for i in ids)/len(ids),len(ids),0)\n        for method in methods:\n            for budget in [0,1,2,3]:\n                metrics=score_state(initial,tuple(range(len(probes))),budget,method)\n                rows.append([name,seed,target,method,budget,*metrics])\n        score_state.cache_clear()\n    tree.cache_clear(); partition.cache_clear()\n    return rows\n\ndef run():\n    rows=[]\n    for name,spec in families().items():\n        for seed in [19,43,73,101,137]:\n            rows.extend(evaluate(name,spec,seed))\n    return rows\n\ndef summarize(rows):\n    grouped=defaultdict(list)\n    for family,seed,target,method,budget,held,probe,n,used in rows:\n        grouped[family,method,budget].append(held)\n    means={k:statistics.mean(v) for k,v in grouped.items()}\n    return {m:{b:statistics.mean(means[f,m,b] for f in families()) for b in [0,1,2,3]}\n            for m in [\'random\',\'random_split\',\'diversity\',\'entropy\',\'lookahead\',\'frugal\']},means\n\nif __name__==\'__main__\':\n    rows=run()\n    with open(\'scores.csv\',\'w\',newline=\'\') as out:\n        w=csv.writer(out);w.writerow([\'family\',\'split\',\'target\',\'policy\',\'budget\',\'heldout_exact\',\'probe_exact\',\'survivors\',\'queries_used\']);w.writerows(rows)\n    print(json.dumps(summarize(rows)[0],indent=2))\n', encoding="utf-8")


In [ ]:
Path('replay.py').write_text('"""Blinded JSON protocol for one episode: python replay.py csv_fields 19 7."""\nimport argparse\nimport json\nimport random\nfrom benchmark import families, outcome\n\ndef episode(family, split, target, seed):\n    configs, smoke, universe, impl=families()[family]\n    if not 0<=target<len(configs):\n        raise ValueError(\'target outside family\')\n    xs=sorted(set(universe)-{smoke})\n    random.Random(split).shuffle(xs)\n    cut=min(16,len(xs)//2)\n    probes,holdout=xs[:cut],xs[cut:]\n    value=outcome(impl,configs[target],smoke)\n    ids=[i for i,c in enumerate(configs) if outcome(impl,c,smoke)==value]\n    random.Random(seed).shuffle(ids)\n    grid=[[outcome(impl,configs[i],x) for x in probes] for i in ids]\n    target_row=ids.index(target)\n    return ids,probes,holdout,grid,target_row,configs,impl\n\ndef play(family, split, target, seed):\n    ids,xs,held,grid,key,configs,impl=episode(family,split,target,seed)\n    print(json.dumps(dict(family=family,inputs=xs,outputs=[[json.loads(o) for o in row] for row in grid],\n                          budget=3,instructions=\'Send {"ask": column} or {"pick": row}; indices start at zero.\')),flush=True)\n    seen=set()\n    for _ in range(4):\n        action=json.loads(input())\n        if set(action)=={\'pick\'}:\n            row=action[\'pick\']\n            if type(row)!=int or not 0<=row<len(ids): raise ValueError(\'invalid row\')\n            correct=all(outcome(impl,configs[ids[row]],x)==outcome(impl,configs[target],x) for x in held)\n            print(json.dumps(dict(heldout_exact=correct,queries=len(seen))),flush=True)\n            return\n        j=action.get(\'ask\')\n        if set(action)!={\'ask\'} or type(j)!=int or not 0<=j<len(xs) or j in seen or len(seen)>=3:\n            raise ValueError(\'invalid or repeated query, or budget exceeded\')\n        seen.add(j)\n        print(json.dumps(dict(answer=json.loads(grid[key][j]))),flush=True)\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'family\',choices=sorted(families()));p.add_argument(\'split\',type=int)\n    p.add_argument(\'target\',type=int);p.add_argument(\'--shuffle\',type=int,default=2026)\n    a=p.parse_args();play(a.family,a.split,a.target,a.shuffle)\n', encoding="utf-8")


In [ ]:
Path('analyze.py').write_text('"""Summarize exact expectations and compute the all-probes ceiling."""\nimport csv, statistics, json\nfrom benchmark import families, outcome\nfrom replay import episode\n\ndef audit():\n    rows=list(csv.DictReader(open(\'scores.csv\')))\n    methods=list(dict.fromkeys(r[\'policy\'] for r in rows))\n    def family_mean(f,m,b,field):\n        return statistics.fmean(float(r[field]) for r in rows if r[\'family\']==f and r[\'policy\']==m and int(r[\'budget\'])==b)\n    report={}\n    for f,spec in families().items():\n        vals=[]\n        for seed in [19,43,73,101,137]:\n            for t in range(len(spec[0])):\n                ids,_,h,grid,key,c,impl=episode(f,seed,t,0)\n                survivors=[i for i,row in zip(ids,grid) if row==grid[key]]\n                vals.append(statistics.fmean(all(outcome(impl,c[i],x)==outcome(impl,c[t],x) for x in h) for i in survivors))\n        report[f]={m:[family_mean(f,m,3,\'heldout_exact\'),family_mean(f,m,3,\'queries_used\')] for m in methods}\n        report[f][\'ceiling\']=statistics.fmean(vals)\n    return report\n\nif __name__==\'__main__\':\n    print(json.dumps(audit(),indent=2))\n', encoding="utf-8")


In [ ]:
Path('intervention.py').write_text('"""Post-hoc regex probe-design intervention. All arms share identical hidden inputs."""\nimport json\nfrom benchmark import evaluate, families\nfrom statistics import fmean\nsingle=[\'q|Q\',\'^q|p\\nq\',\'q.z|q\\nz\',r\'\\w|Ж\']\ncompound=[\'^q.\\\\w|q0a\\nQ0b\\nq\\nЖ\\nq0é\\nQ\\nz\',\'^q.\\\\w|Q0Ж\\nq\\nx\\nq0b\',\n          \'^q.\\\\w|q\\nЖ\\nQ0b\\nq1Ж\',\'^q.\\\\w|Q0b\\nq\\ny\\nq1Ж\']\ndef transfer_suite():\n    # Separate vocabulary, strings and patterns; fixed after the diagnostic analysis.\n    return [p+\'|\'+s for p in [r"^r.\\w",r"r.\\d","^r","r.t",r"\\w+",r"\\d+"]\n            for s in ["R1Ж","r\\nЖ","u\\nr0a","Ｒ","r0۷","r\\nt","!R0z","r0x\\nR1Ж"]]\n\ndef study(transfer=False):\n    out={}\n    for label,probes in [(\'original\',[]),(\'single\',single),(\'compound\',compound)]:\n        rows=[r for seed in [19,43,73,101,137] for r in evaluate(\'regex_search\',families()[\'regex_search\'],seed,probes,transfer_suite() if transfer else None)]\n        out[label]={b:fmean(r[5] for r in rows if r[3]==\'frugal\' and r[4]==b) for b in (1,2,3)}\n    return out\nif __name__==\'__main__\':\n    print(json.dumps({"original_holdout":study(),"transfer":study(True)},indent=2))\n', encoding="utf-8")


In [ ]:
Path('verify.py').write_text('"""Independent decision-tree and blind-grader checks."""\nimport csv, json, subprocess, sys\nfrom fractions import Fraction\nfrom itertools import combinations\nfrom benchmark import families, evaluate\nfrom replay import episode\nfrom intervention import study\n\ndef check():\n    # Exhaustive two-query search using explicit leaf lists, independently of the DP.\n    for name in families():\n        ids,xs,held,grid,key,c,impl=episode(name,19,0,99)\n        assert len(ids)==len(c)\n        best=0\n        for j in range(len(xs)):\n            groups={}\n            for i,row in enumerate(grid):\n                groups.setdefault(row[j],[]).append(i)\n            leaves=sum(max(len({grid[i][k] for i in group}) for k in range(len(xs))) for group in groups.values())\n            best=max(best,leaves)\n        # Uniform selection in each leaf scores exactly one correct identity per leaf.\n        out=evaluate(name,families()[name],19)\n        from math import isclose\n        vals=[1/r[7] for r in out if r[3]==\'frugal\' and r[4]==2]\n        assert isclose(sum(vals),best),name\n        for b in (1,2,3):\n            a=[r for r in out if r[3]==\'lookahead\' and r[4]==b]\n            z=[r for r in out if r[3]==\'frugal\' and r[4]==b]\n            assert isclose(sum(1/r[7] for r in a),sum(1/r[7] for r in z))\n            assert sum(r[8] for r in z)<=sum(r[8] for r in a)+1e-9\n    ids,_,_,_,key,_,_=episode(\'csv_fields\',19,7,2026)\n    cmd=[sys.executable,\'replay.py\',\'csv_fields\',\'19\',\'7\']\n    result=subprocess.run(cmd,input=\'{"ask":0}\\n\'+json.dumps({\'pick\':key})+\'\\n\',capture_output=True,text=True,check=True)\n    assert json.loads(result.stdout.splitlines()[-1])=={\'heldout_exact\':True,\'queries\':1}\n    print(\'Eight independent optimality checks and blind protocol check passed.\')\n\nif __name__==\'__main__\':check()\n', encoding="utf-8")


In [ ]:
Path('export_cases.py').write_text('"""Export exact query/held-out matrices for replay across Python versions."""\nimport gzip,json,random\nfrom benchmark import families,outcome\ndef export(path):\n    with gzip.open(path,\'wt\',encoding=\'utf8\') as dst:\n        for f,(configs,smoke,universe,impl) in families().items():\n            for seed in [19,43,73,101,137]:\n                xs=sorted(set(universe)-{smoke})\n                random.Random(seed).shuffle(xs)\n                cut=min(16,len(xs)//2)\n                def matrix(inputs):\n                    return [[json.loads(outcome(impl,c,x)) for x in inputs] for c in configs]\n                record=dict(family=f,split=seed,smoke=smoke,configs=[list(map(repr,c)) for c in configs],\n                            query_inputs=xs[:cut],query_outputs=matrix(xs[:cut]),\n                            heldout_inputs=xs[cut:],heldout_outputs=matrix(xs[cut:]))\n                dst.write(json.dumps(record,ensure_ascii=True)+\'\\n\')\nif __name__==\'__main__\': export(\'frozen_cases.jsonl.gz\')\n', encoding="utf-8")


In [ ]:
Path('cognitive_probe.py').write_text('"""Replay frozen evidence with semantic labels visible or masked. Python standard library."""\nimport argparse\nfrom collections import Counter\nimport gzip\nimport json\nimport math\nfrom pathlib import Path\nimport random\n\n\ndef mask(grid):\n    result = [[] for row in grid]\n    for column in zip(*grid):\n        labels = {}\n        for row, value in enumerate(column):\n            result[row].append(labels.setdefault(value, f"v{len(labels)}"))\n    return result\n\n\ndef choose(grid, survivors, remaining):\n    def entropy(j):\n        counts = Counter(grid[i][j] for i in survivors)\n        return -sum((n / len(survivors)) * math.log2(n / len(survivors)) for n in counts.values())\n    best = max(remaining, key=entropy)\n    return best if entropy(best) > 0 else None\n\n\ndef run():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--check\', action=\'store_true\')\n    parser.add_argument(\'--family\', default=\'csv_fields\')\n    parser.add_argument(\'--split\', type=int, default=19)\n    parser.add_argument(\'--target\', type=int, default=0)\n    parser.add_argument(\'--view\', choices=[\'named\', \'masked\'], default=\'masked\')\n    args = parser.parse_args()\n    with gzip.open(Path(__file__).with_name(\'frozen_cases.jsonl.gz\'), \'rt\') as f:\n        cases = [json.loads(line) for line in f]\n    count = 0\n    for case in cases:\n        if not args.check and (case[\'family\'], case[\'split\']) != (args.family, args.split):\n            continue\n        order = list(range(len(case[\'configs\'])))\n        random.Random(2026 + case[\'split\']).shuffle(order)\n        named = [[json.dumps(v) for v in case[\'query_outputs\'][i]] for i in order]\n        masked = mask(named)\n        for target in range(len(order)) if args.check else [args.target]:\n            key = order.index(target)\n            if args.check:\n                survivors = list(range(len(order)))\n                remaining = list(range(len(named[0])))\n                for _ in range(3):\n                    j = choose(named, survivors, remaining)\n                    assert j == choose(masked, survivors, remaining)\n                    if j is None:\n                        break\n                    remaining.remove(j)\n                    survivors = [i for i in survivors if named[i][j] == named[key][j]]\n                    assert survivors == [i for i in range(len(order)) if all(masked[i][q] == masked[key][q] for q in set(range(len(named[0]))) - set(remaining))]\n                count += 1\n                continue\n            grid = named if args.view == \'named\' else masked\n            print(json.dumps(dict(inputs=case[\'query_inputs\'] if args.view==\'named\' else list(range(len(grid[0]))), outputs=grid, budget=3, instructions=\'Send {"ask": column} or {"pick": row}. Indices start at zero.\')), flush=True)\n            asked = set()\n            while True:\n                action = json.loads(input())\n                if set(action)=={\'pick\'} and type(action[\'pick\']) is int and 0<=action[\'pick\']<len(order):\n                    print(json.dumps(dict(heldout_exact=case[\'heldout_outputs\'][order[action[\'pick\']]]==case[\'heldout_outputs\'][target], questions=len(asked))))\n                    return\n                j = action.get(\'ask\')\n                if set(action)!={\'ask\'} or type(j) is not int or j in asked or len(asked)>=3 or not 0<=j<len(grid[0]):\n                    raise ValueError(\'Invalid action\')\n                asked.add(j)\n                print(json.dumps(dict(answer=grid[key][j])), flush=True)\n    if args.check:\n        print(f\'{count} target-split episodes passed the paired entropy decision and survivor checks.\')\n    else:\n        parser.error(\'Unknown family or split\')\n\n\nif __name__ == \'__main__\':\n    run()\n', encoding="utf-8")


In [ ]:
Path('paper_analysis.py').write_text('"""Rebuild revision tables and selected-transcript costs from the benchmark.\n\nCPU only. Run beside scores.csv. Original benchmark methods are unchanged.\nCosts count Unicode code points in selected input and serialized outcome strings,\nexcluding protocol wrappers, JSON actions, repeated prompt context and thinking.\n"""\nimport csv\nimport hashlib\nimport json\nimport math\nfrom collections import Counter, defaultdict\nfrom functools import lru_cache\nfrom pathlib import Path\nimport random\nimport statistics\nimport sys\n\nfrom benchmark import evaluate, families, outcome\nfrom intervention import compound, single, transfer_suite\n\n\ndef trajectory(query, target, policy, budget):\n    @lru_cache(None)\n    def parts(ids, j):\n        groups = defaultdict(list)\n        for i in ids:\n            groups[query[i][j]].append(i)\n        return tuple(tuple(g) for g in groups.values())\n\n    @lru_cache(None)\n    def tree(ids, depth):\n        best = (1, 0, None)\n        if depth:\n            for j in range(len(query[0])):\n                groups = parts(ids, j)\n                if len(groups) < 2:\n                    continue\n                children = [tree(g, depth - 1) for g in groups]\n                leaves = sum(x[0] for x in children)\n                cost = len(ids) + sum(x[1] for x in children)\n                if leaves > best[0] or leaves == best[0] and cost < best[1]:\n                    best = (leaves, cost, j)\n        return best\n\n    ids = tuple(range(len(query)))\n    remaining = list(range(len(query[0])))\n    asked, information = [], []\n    for depth in range(budget, 0, -1):\n        choices = [j for j in remaining if len(parts(ids, j)) > 1]\n        if not choices:\n            break\n        j = tree(ids, depth)[2] if policy == \'frugal\' else max(choices, key=lambda j: -sum(len(g) * math.log2(len(g)) for g in parts(ids, j)))\n        information.append(-sum((len(g) / len(ids)) * math.log2(len(g) / len(ids)) for g in parts(ids, j)))\n        asked.append(j)\n        remaining.remove(j)\n        ids = next(g for g in parts(ids, j) if target in g)\n    return ids, asked, information\n\n\ndef build(output):\n    output.mkdir(parents=True, exist_ok=True)\n    rows = list(csv.DictReader(Path(\'scores.csv\').open(encoding=\'utf-8\', newline=\'\')))\n    policies = [\'random\', \'random_split\', \'diversity\', \'entropy\', \'lookahead\', \'frugal\']\n    means, by_family = {}, {}\n    for p in policies:\n        means[p] = {}\n        for b in range(4):\n            groups = {f: [r for r in rows if r[\'family\'] == f and r[\'policy\'] == p and int(r[\'budget\']) == b] for f in families()}\n            means[p][b] = {k: statistics.fmean(statistics.fmean(float(r[k]) for r in group) for group in groups.values()) for k in [\'heldout_exact\', \'queries_used\']}\n            if b == 3:\n                by_family[p] = {f: statistics.fmean(float(r[\'heldout_exact\']) for r in group) for f, group in groups.items()}\n    sensitivity = []\n    for f in families():\n        difference = statistics.fmean(by_family[\'frugal\'][g] - by_family[\'entropy\'][g] for g in families() if g != f)\n        sensitivity.append({\'excluded_family\': f, \'frugal_minus_entropy_pp\': 100 * difference})\n    classes = []\n    import gzip\n    for line in gzip.open(Path(__file__).with_name(\'frozen_cases.jsonl.gz\'), \'rt\', encoding=\'utf-8\'):\n        c = json.loads(line)\n        q = [json.dumps(v, sort_keys=True) for v in c[\'query_outputs\']]\n        h = [json.dumps(v, sort_keys=True) for v in c[\'heldout_outputs\']]\n        groups = defaultdict(list)\n        for i, signature in enumerate(q):\n            groups[signature].append(i)\n        uniform = sum(sum(n * n for n in Counter(h[i] for i in group).values()) / len(group) for group in groups.values()) / len(q)\n        optimal = sum(max(Counter(h[i] for i in group).values()) for group in groups.values()) / len(q)\n        classes.append({\'family\': c[\'family\'], \'split\': c[\'split\'], \'configurations\': len(q), \'query_classes\': len(groups), \'joint_classes\': len(set(zip(q, h))), \'all_query_uniform\': uniform, \'evaluator_only_majority\': optimal})\n\n    config, smoke, universe, impl = families()[\'regex_search\']\n    raw, crossed = [], []\n    capacities = []\n    for pool, replacement in [(\'original\', []), (\'single\', single), (\'compound\', compound)]:\n        for split in [19, 43, 73, 101, 137]:\n            xs = sorted(set(universe) - {smoke})\n            random.Random(split).shuffle(xs)\n            cut = min(16, len(xs) // 2)\n            probes, held = xs[:cut], xs[cut:]\n            if replacement:\n                probes = probes[:-len(replacement)] + replacement\n            query = [tuple(outcome(impl, c, x) for x in probes) for c in config]\n            capacities.extend({\'pool\': pool, \'split\': split, \'column\': j, \'outcomes\': len(Counter(row[j] for row in query)), \'initial_answer_entropy_bits\': -sum((n / len(config)) * math.log2(n / len(config)) for n in Counter(row[j] for row in query).values())} for j in range(len(probes)))\n            for suite, inputs in [(\'original_holdout\', held), (\'transfer\', transfer_suite())]:\n                hidden = [tuple(outcome(impl, c, x) for x in inputs) for c in config]\n                reference = evaluate(\'regex_search\', families()[\'regex_search\'], split, replacement, inputs)\n                keyed = {(r[2], r[3], r[4]): r for r in reference}\n                for policy in [\'entropy\', \'frugal\']:\n                    for budget in [1, 2, 3]:\n                        for target in range(len(config)):\n                            ids, asked, information = trajectory(query, target, policy, budget)\n                            success = sum(hidden[i] == hidden[target] for i in ids) / len(ids)\n                            expected = keyed[target, policy, budget]\n                            assert abs(success - expected[5]) < 1e-12 and len(asked) == expected[8]\n                            raw.append({\'suite\': suite, \'pool\': pool, \'split\': split, \'target\': target, \'policy\': policy, \'budget\': budget, \'heldout_exact\': success, \'questions\': len(asked), \'input_characters\': sum(len(probes[j]) for j in asked), \'answer_characters\': sum(len(query[target][j]) for j in asked), \'conditional_information_bits\': sum(information), \'asked\': asked})\n        for suite in [\'original_holdout\', \'transfer\']:\n            for p in [\'entropy\', \'frugal\']:\n                for b in [1, 2, 3]:\n                    rs = [r for r in raw if (r[\'suite\'], r[\'pool\'], r[\'policy\'], r[\'budget\']) == (suite, pool, p, b)]\n                    crossed.append({\'suite\': suite, \'pool\': pool, \'policy\': p, \'budget\': b, \'episodes\': len(rs), **{k: statistics.fmean(r[k] for r in rs) for k in [\'heldout_exact\', \'questions\', \'input_characters\', \'answer_characters\', \'conditional_information_bits\']}})\n    result = {\'runtime\': sys.version, \'scope\': \'Descriptive analysis of existing benchmark and exploratory intervention. No new family or model experiment.\', \'score_rows\': len(rows), \'score_sha256\': hashlib.sha256(Path(\'scores.csv\').read_bytes()).hexdigest(), \'baseline\': means, \'per_family_budget3\': by_family, \'leave_one_family_out\': sensitivity, \'behavior_classes\': classes, \'crossed\': crossed, \'pool_capacity\': capacities, \'cost_definition\': __doc__}\n    (output / \'revision-results.json\').write_text(json.dumps(result, indent=2) + \'\\n\', encoding=\'utf-8\')\n    with (output / \'selected-transcripts.csv\').open(\'w\', encoding=\'utf-8\', newline=\'\') as f:\n        writer = csv.DictWriter(f, fieldnames=list(raw[0]))\n        writer.writeheader()\n        writer.writerows({**r, \'asked\': json.dumps(r[\'asked\'])} for r in raw)\n    table = [\'# Generated revision tables\', \'\', \'| Regex pool / policy | Agreement (%) | Questions | Input characters | Answer characters |\', \'|---|---:|---:|---:|---:|\']\n    for r in crossed:\n        if r[\'suite\'] == \'original_holdout\' and r[\'budget\'] == 3:\n            table.append(f"| {r[\'pool\']} / {r[\'policy\']} | {100*r[\'heldout_exact\']:.2f} | {r[\'questions\']:.4f} | {r[\'input_characters\']:.3f} | {r[\'answer_characters\']:.3f} |")\n    (output / \'TABLES.md\').write_text(\'\\n\'.join(table) + \'\\n\', encoding=\'utf-8\')\n    print(\'\\n\'.join(table))\n    return result\n\n\nif __name__ == \'__main__\':\n    import argparse\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--output\', type=Path, default=Path(\'revision-results\'))\n    build(parser.parse_args().output)\n', encoding="utf-8")


In [ ]:
Path('pilot.py').write_text('"""Strict frozen-matrix supervisor. No learned-model results are bundled.\n\nCore commands and tests use the standard library. Live inference additionally\nneeds a local Hugging Face tokenizer and a local chat-completions HTTP server.\nThe evaluated completion model receives text only and has no execution tools.\n"""\nimport argparse\nimport copy\nfrom collections import Counter\nfrom datetime import datetime, timezone\nfrom functools import lru_cache\nimport gzip\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport random\nimport secrets\nimport statistics\nimport time\nfrom urllib.request import Request, urlopen, build_opener, ProxyHandler\nfrom urllib.parse import urlparse\n\nINSTRUCTION = \'Select a candidate from the displayed behavior matrix. Send exactly one JSON object: {"ask": column} or {"pick": row}. Indices start at zero. Each ask returns the intended output. Use at most three distinct asks, then pick a row consistent with every answer. You may pick early. No prose or other keys.\'\n\n\ndef encode(value):\n    return json.dumps(value, sort_keys=True, separators=(\',\', \':\'), ensure_ascii=True)\n\n\ndef sha(data):\n    return hashlib.sha256(data).hexdigest()\n\n\ndef load_cases(path):\n    with gzip.open(path, \'rt\', encoding=\'utf-8\') as f:\n        return {(c[\'family\'], c[\'split\']): c for c in map(json.loads, f)}\n\n\nclass ProtocolError(ValueError):\n    pass\n\n\ndef parse_action(raw):\n    def unique(items):\n        obj = {}\n        for key, value in items:\n            if key in obj:\n                raise ProtocolError(\'duplicate_key\')\n            obj[key] = value\n        return obj\n    try:\n        action = json.loads(raw, object_pairs_hook=unique)\n    except (ValueError, TypeError) as exc:\n        raise ProtocolError(\'malformed_json\') from exc\n    if type(action) is not dict or len(action) != 1 or next(iter(action)) not in {\'ask\', \'pick\'}:\n        raise ProtocolError(\'invalid_keys\')\n    if type(next(iter(action.values()))) is not int:\n        raise ProtocolError(\'noninteger_index\')\n    return action\n\n\nclass Session:\n    def __init__(self, case, target, order, view, label_seed=0, budget=3):\n        if sorted(order) != list(range(len(case[\'configs\']))) or not 0 <= target < len(order):\n            raise ValueError(\'invalid private episode\')\n        self.case, self.target, self.order = case, target, order\n        self.key = order.index(target)\n        named = [[encode(v) for v in case[\'query_outputs\'][i]] for i in order]\n        self.grid = copy.deepcopy(named)\n        if view == \'masked\':\n            rng = random.Random(label_seed)\n            for j in range(len(named[0])):\n                values = list(dict.fromkeys(row[j] for row in named))\n                labels = [f\'v{k}\' for k in range(len(values))]\n                rng.shuffle(labels)\n                mapping = dict(zip(values, labels))\n                for i in range(len(order)):\n                    self.grid[i][j] = mapping[named[i][j]]\n        elif view != \'named\':\n            raise ValueError(\'unknown view\')\n        self.budget = budget\n        self.asked = []\n        self.survivors = list(range(len(order)))\n        self.finished = False\n        self.initial = {\'inputs\': case[\'query_inputs\'] if view == \'named\' else list(range(len(named[0]))), \'outputs\': self.grid, \'budget\': budget, \'instructions\': INSTRUCTION if budget else \'Choose now. Send exactly one JSON object with a pick integer. No questions are available.\'}\n\n    def step(self, action):\n        if self.finished:\n            raise ProtocolError(\'already_finished\')\n        action = parse_action(encode(action))\n        if \'ask\' in action:\n            j = action[\'ask\']\n            if not 0 <= j < len(self.grid[0]):\n                raise ProtocolError(\'query_out_of_range\')\n            if j in self.asked:\n                raise ProtocolError(\'repeated_query\')\n            if len(self.asked) >= self.budget:\n                raise ProtocolError(\'budget_exceeded\')\n            self.asked.append(j)\n            self.survivors = [i for i in self.survivors if self.grid[i][j] == self.grid[self.key][j]]\n            return {\'answer\': self.grid[self.key][j]}, None\n        row = action[\'pick\']\n        if not 0 <= row < len(self.order):\n            raise ProtocolError(\'pick_out_of_range\')\n        self.finished = True\n        held = self.case[\'heldout_outputs\'][self.order[row]] == self.case[\'heldout_outputs\'][self.target]\n        consistent = row in self.survivors\n        grade = {\'status\': \'completed\', \'valid_completion\': True, \'query_consistent\': consistent, \'heldout_exact\': held, \'success\': consistent and held, \'identity_exact\': self.order[row] == self.target, \'questions\': len(self.asked), \'survivors\': len(self.survivors)}\n        return None, grade  # The private grade is never returned to the model.\n\n\ndef baseline(session, policy):\n    grid, target = session.grid, session.key\n    @lru_cache(None)\n    def visit(ids, remaining, depth):\n        choices = [j for j in remaining if len({grid[i][j] for i in ids}) > 1] if depth else []\n        if not choices:\n            held = session.case[\'heldout_outputs\']\n            score = sum(held[session.order[i]] == held[session.target] for i in ids) / len(ids)\n            return score, 0.0\n        if policy == \'entropy\':\n            def utility(j):\n                from collections import Counter\n                return -sum(n * math.log2(n) for n in Counter(grid[i][j] for i in ids).values())\n            choices = [max(choices, key=utility)]\n        values = [visit(tuple(i for i in ids if grid[i][j] == grid[target][j]), tuple(k for k in remaining if k != j), depth - 1) for j in choices]\n        return statistics.fmean(v[0] for v in values), 1 + statistics.fmean(v[1] for v in values)\n    if policy not in {\'entropy\', \'random_split\'}:\n        raise ValueError(\'unknown policy\')\n    score, questions = visit(tuple(range(len(grid))), tuple(range(len(grid[0]))), session.budget)\n    return {\'heldout_exact\': score, \'success\': score, \'questions\': questions, \'final_selection\': \'uniform among consistent survivors\'}\n\n\ndef make_plan(cases, data_hash, info, seed=None, controls=False):\n    required = {\'served_model\', \'model_revision\', \'tokenizer_revision\', \'quantization\', \'serving_stack\', \'hardware\', \'context_tokens\', \'max_tokens\', \'temperature\', \'top_p\', \'seed\', \'timeout_seconds\'}\n    if set(info) - {\'chat_template_kwargs\'} != required or any(value is None for value in info.values()):\n        raise ValueError(\'model-info must contain exactly the documented fields with confirmed values\')\n    if any(type(info[k]) is not int or info[k] <= 0 for k in [\'context_tokens\', \'max_tokens\', \'timeout_seconds\']):\n        raise ValueError(\'invalid limits\')\n    rng = random.Random(secrets.randbits(256) if seed is None else seed)\n    episodes = []\n    for f in sorted({f for f, split in cases if split == 19}):\n        c = cases[f, 19]\n        n = len(c[\'configs\'])\n        # One secret draw from each half of configuration order, per family.\n        for target in [rng.randrange(n // 2), rng.randrange(n // 2, n)]:\n            order = list(range(n))\n            rng.shuffle(order)\n            episodes.append({\'pair_id\': f\'pair-{len(episodes):02}\', \'family\': f, \'split\': 19, \'target\': target, \'order\': order, \'label_seed\': rng.getrandbits(64)})\n    rng.shuffle(episodes)\n    schedule = []\n    for i, episode in enumerate(episodes):\n        for view in ([\'named\', \'masked\'] if i % 2 == 0 else [\'masked\', \'named\']):\n            schedule.append({**episode, \'view\': view, \'session_id\': episode[\'pair_id\'] + \'-\' + view})\n    if controls:\n        extra = []\n        for entry in schedule:\n            if entry[\'view\'] == \'named\':\n                for arm in (\'zero\', \'entropy_pick\'):\n                    extra.append({**entry, \'arm\': arm, \'session_id\': entry[\'pair_id\'] + \'-\' + arm})\n        rng.shuffle(extra)\n        schedule.extend(extra)\n    return {\'schema\': 2, \'created_utc\': datetime.now(timezone.utc).isoformat(), \'data_sha256\': data_hash, \'source_sha256\': sha(Path(__file__).read_bytes()), \'instruction_sha256\': sha(INSTRUCTION.encode()), \'model_info\': info, \'retry_rule\': \'No retries. Preserve all failures; corrected infrastructure requires a new run ID.\', \'sampling\': \'Two targets per family, stratified by configuration-index half; convenience feasibility sample, no population inference.\', \'schedule\': schedule}\n\n\ndef model_request(messages, info):\n    # Explicit whitelist. No plan, target, grading data, tools or filenames.\n    request = {\'model\': info[\'served_model\'], \'messages\': copy.deepcopy(messages), \'temperature\': info[\'temperature\'], \'top_p\': info[\'top_p\'], \'seed\': info[\'seed\'], \'max_tokens\': info[\'max_tokens\'], \'stream\': False}\n    if info.get(\'chat_template_kwargs\'):\n        request[\'chat_template_kwargs\'] = copy.deepcopy(info[\'chat_template_kwargs\'])\n    return request\n\n\ndef summarize(plan, records):\n    allowed = {s[\'session_id\'] for s in plan[\'schedule\']}\n    if len({r[\'session_id\'] for r in records}) != len(records) or any(r[\'session_id\'] not in allowed for r in records):\n        raise ValueError(\'duplicate or unscheduled result\')\n    keyed = {r[\'session_id\']: r for r in records}\n    output = {\'scheduled\': len(allowed), \'recorded\': len(records), \'unstarted_or_interrupted\': len(allowed) - len(records), \'conditions\': {}}\n    for view in [\'named\', \'masked\']:\n        ids = [s[\'session_id\'] for s in plan[\'schedule\'] if s[\'view\'] == view and s.get(\'arm\', \'free\') == \'free\']\n        output[\'conditions\'][view] = {\'denominator\': len(ids), \'successes\': sum(bool(keyed.get(i, {}).get(\'success\', False)) for i in ids), \'heldout_only_successes\': sum(bool(keyed.get(i, {}).get(\'heldout_exact\', False)) for i in ids), \'completed\': sum(keyed.get(i, {}).get(\'status\') == \'completed\' for i in ids)}\n    pairs = {}\n    for entry in plan[\'schedule\']:\n        if entry.get(\'arm\', \'free\') == \'free\':\n            pairs.setdefault(entry[\'pair_id\'], {})[entry[\'view\']] = keyed.get(entry[\'session_id\'])\n    paired = {\'named_wins\': 0, \'masked_wins\': 0, \'ties\': 0, \'incomplete_pairs\': 0}\n    for pair in pairs.values():\n        if any(pair.get(v) is None for v in [\'named\', \'masked\']):\n            paired[\'incomplete_pairs\'] += 1\n        a, b = [bool((pair.get(v) or {}).get(\'success\', False)) for v in [\'named\', \'masked\']]\n        paired[\'ties\' if a == b else \'named_wins\' if a else \'masked_wins\'] += 1\n    output[\'paired_failure_inclusive\'] = paired\n    complete = {\'named_wins\': 0, \'masked_wins\': 0, \'ties\': 0, \'pairs\': 0}\n    for pair in pairs.values():\n        if all(pair.get(v) is not None for v in (\'named\', \'masked\')):\n            a, b = [bool(pair[v].get(\'success\')) for v in (\'named\', \'masked\')]\n            complete[\'pairs\'] += 1\n            complete[\'ties\' if a == b else \'named_wins\' if a else \'masked_wins\'] += 1\n    output[\'paired_complete\'] = complete\n    output[\'arms\'] = {}\n    for arm in (\'free\', \'zero\', \'entropy_pick\'):\n        ids = [e[\'session_id\'] for e in plan[\'schedule\'] if e.get(\'arm\', \'free\') == arm]\n        output[\'arms\'][arm] = {\'scheduled\': len(ids), \'successes\': sum(bool(keyed.get(i, {}).get(\'success\')) for i in ids), \'statuses\': dict(Counter(keyed.get(i, {}).get(\'status\', \'unstarted\') for i in ids))}\n    return output\n\n\ndef save_json(path, value):\n    with path.open(\'x\', encoding=\'utf-8\') as f:\n        f.write(json.dumps(value, indent=2) + \'\\n\')\n        f.flush()\n        os.fsync(f.fileno())\n\n\ndef count_tokens(tokenizer, messages, info):\n    encoded = tokenizer.apply_chat_template(messages, tokenize=True,\n        add_generation_prompt=True, return_dict=False, truncation=False,\n        **info.get(\'chat_template_kwargs\', {}))\n    ids = encoded[\'input_ids\'] if hasattr(encoded, \'keys\') else encoded\n    if hasattr(ids, \'tolist\'):\n        ids = ids.tolist()\n    if ids and isinstance(ids[0], list):\n        if len(ids) != 1:\n            raise ValueError(\'expected one tokenized conversation\')\n        ids = ids[0]\n    return len(ids)\n\n\ndef new_session(cases, spec):\n    return Session(cases[spec[\'family\'], spec[\'split\']], spec[\'target\'],\n                   spec[\'order\'], spec[\'view\'], spec[\'label_seed\'],\n                   budget=0 if spec.get(\'arm\') == \'zero\' else 3)\n\n\ndef entropy_question(session):\n    choices = [j for j in range(len(session.grid[0])) if j not in session.asked]\n    def utility(j):\n        sizes = Counter(session.grid[i][j] for i in session.survivors).values()\n        return -sum(n * math.log2(n) for n in sizes)\n    informative = [j for j in choices if len({session.grid[i][j] for i in session.survivors}) > 1]\n    return max(informative, key=utility) if informative else None\n\n\ndef initial_messages(session):\n    return [{\'role\': \'system\', \'content\': session.initial[\'instructions\']},\n            {\'role\': \'user\', \'content\': encode(session.initial)}]\n\n\ndef preflight(plan, cases, tokenizer):\n    info = plan[\'model_info\']\n    report = []\n    for spec in plan[\'schedule\']:\n        session = new_session(cases, spec)\n        messages = initial_messages(session)\n        first = count_tokens(tokenizer, messages, info)\n        # Conservative history: all permitted turns use the largest rendered\n        # query/action/answer exchange for this matrix. No truncation is used.\n        exchanges = [(encode({\'ask\': j}), encode({\'answer\': row[j]}))\n                     for row in session.grid for j in range(len(row))]\n        action, answer = max(exchanges, key=lambda e: count_tokens(tokenizer,\n            messages + [{\'role\': \'assistant\', \'content\': e[0]},\n                        {\'role\': \'user\', \'content\': e[1]}], info))\n        for _ in range(session.budget):\n            messages.extend([{\'role\': \'assistant\', \'content\': action}, {\'role\': \'user\', \'content\': answer}])\n        worst = count_tokens(tokenizer, messages, info)\n        report.append({\'session_id\': spec[\'session_id\'], \'opening_tokens\': first,\n                       \'conservative_tokens\': worst, \'fits\': worst + info[\'max_tokens\'] <= info[\'context_tokens\']})\n    return report\n\n\ndef replay(directory, data_path):\n    plan = json.loads((directory / \'plan.json\').read_text())\n    if sha(data_path.read_bytes()) != plan[\'data_sha256\']:\n        raise ValueError(\'data hash mismatch\')\n    cases = load_cases(data_path)\n    records = []\n    for spec in plan[\'schedule\']:\n        path = directory / (spec[\'session_id\'] + \'.result.json\')\n        if not path.exists():\n            continue\n        record = json.loads(path.read_text())\n        session = new_session(cases, spec)\n        grade = None\n        questions = []\n        usage = Counter()\n        for event in map(json.loads, (directory / record[\'transcript\']).read_text().splitlines()):\n            if event.get(\'usage\'):\n                usage.update({k: v for k, v in event[\'usage\'].items() if isinstance(v, (int, float))})\n            if \'action\' in event:\n                before = len(session.survivors)\n                def query_entropy(j):\n                    sizes = Counter(session.grid[i][j] for i in session.survivors).values()\n                    return -sum((n / before) * math.log2(n / before) for n in sizes)\n                question = event[\'action\'].get(\'ask\')\n                entropy = query_entropy(question) if question is not None else None\n                best = max((query_entropy(j) for j in range(len(session.grid[0])) if j not in session.asked), default=0)\n                _, grade = session.step(event[\'action\'])\n                if \'ask\' in event[\'action\']:\n                    questions.append({\'column\': event[\'action\'][\'ask\'], \'survivors_before\': before, \'survivors_after\': len(session.survivors), \'uninformative\': before == len(session.survivors), \'fraction_eliminated\': 1 - len(session.survivors) / before, \'entropy_bits\': entropy, \'best_available_entropy_bits\': best, \'relative_entropy\': entropy / best if best else None})\n        hidden = session.case[\'heldout_outputs\']\n        record[\'uniform_survivor_score\'] = sum(hidden[session.order[i]] == hidden[session.target] for i in session.survivors) / len(session.survivors)\n        record[\'survivors_after_questions\'] = len(session.survivors)\n        record[\'question_diagnostics\'] = questions\n        record[\'token_usage\'] = dict(usage)\n        if record[\'status\'] == \'completed\':\n            if grade is None or any(record.get(k) != v for k, v in grade.items()):\n                raise ValueError(\'transcript grade mismatch: \' + spec[\'session_id\'])\n        records.append(record)\n    result = summarize(plan, records)\n    result[\'transcript_diagnostics\'] = [{k: r[k] for k in (\'session_id\', \'uniform_survivor_score\', \'survivors_after_questions\', \'question_diagnostics\', \'token_usage\')} for r in records]\n    return result\n\n\ndef run(plan_path, data_path, endpoint, tokenizer_path, output):\n    # Local endpoints only. No authenticated service or paid API is used.\n    parsed = urlparse(endpoint)\n    if parsed.scheme != \'http\' or parsed.hostname not in {\'127.0.0.1\', \'localhost\', \'::1\'} or parsed.username or parsed.password or parsed.query or parsed.fragment:\n        raise ValueError(\'Use a local unauthenticated server origin\')\n    plan_bytes = plan_path.read_bytes()\n    plan = json.loads(plan_bytes)\n    if sha(data_path.read_bytes()) != plan[\'data_sha256\'] or sha(Path(__file__).read_bytes()) != plan[\'source_sha256\']:\n        raise ValueError(\'frozen data or supervisor changed\')\n    from transformers import AutoTokenizer\n    tokenizer = AutoTokenizer.from_pretrained(str(tokenizer_path), local_files_only=True, trust_remote_code=False)\n    output.mkdir(parents=True, exist_ok=False)\n    save_json(output / \'plan.json\', plan)\n    save_json(output / \'RUN.json\', {\'plan_sha256\': sha(plan_bytes), \'endpoint\': endpoint, \'tokenizer_path\': str(tokenizer_path), \'scope\': \'Text-only frozen-matrix evaluation; preserve all failures and raw responses.\'})\n    cases = load_cases(data_path)\n    records = []\n    info = plan[\'model_info\']\n    checks = preflight(plan, cases, tokenizer)\n    save_json(output / \'PREFLIGHT.json\', checks)\n    if not all(r[\'fits\'] for r in checks):\n        raise ValueError(\'context preflight failed; inspect PREFLIGHT.json\')\n    opener = build_opener(ProxyHandler({}))\n    for spec in plan[\'schedule\']:\n        session = new_session(cases, spec)\n        messages = initial_messages(session)\n        started = time.monotonic()\n        row = {\'session_id\': spec[\'session_id\'], \'view\': spec[\'view\'], \'status\': \'interrupted\', \'success\': False, \'heldout_exact\': None}\n        event_path = output / (spec[\'session_id\'] + \'.events.jsonl\')\n        try:\n            with event_path.open(\'x\', encoding=\'utf-8\') as events:\n                def log(event):\n                    events.write(encode(event) + \'\\n\')\n                    events.flush()\n                    os.fsync(events.fileno())\n                if spec.get(\'arm\') == \'entropy_pick\':\n                    for _ in range(3):\n                        j = entropy_question(session)\n                        if j is None:\n                            break\n                        action = {\'ask\': j}\n                        answer, _ = session.step(action)\n                        log({\'action\': action, \'source\': \'entropy\', \'answer\': answer})\n                        messages.extend([{\'role\': \'assistant\', \'content\': encode(action)}, {\'role\': \'user\', \'content\': encode(answer)}])\n                    messages.append({\'role\': \'user\', \'content\': \'Select a consistent row now. Send only a JSON pick object.\'})\n                for turn in range(1 if spec.get(\'arm\') in (\'zero\', \'entropy_pick\') else 4):\n                    request = model_request(messages, info)\n                    count = count_tokens(tokenizer, messages, info)\n                    log({\'turn\': turn, \'request\': request, \'prompt_tokens_preflight\': count})\n                    if count + info[\'max_tokens\'] > info[\'context_tokens\']:\n                        raise ProtocolError(\'context_overflow\')\n                    remaining = info[\'timeout_seconds\'] - (time.monotonic() - started)\n                    if remaining <= 0:\n                        raise TimeoutError(\'session_timeout\')\n                    with opener.open(Request(endpoint.rstrip(\'/\') + \'/v1/chat/completions\', data=encode(request).encode(), headers={\'Content-Type\': \'application/json\'}), timeout=remaining) as response:\n                        raw = response.read().decode(\'utf-8\')\n                    log({\'turn\': turn, \'raw_response\': raw})\n                    response = json.loads(raw)\n                    choice = response[\'choices\'][0]\n                    if choice[\'finish_reason\'] == \'length\':\n                        raise ProtocolError(\'output_truncated\')\n                    if choice[\'message\'].get(\'tool_calls\'):\n                        raise ProtocolError(\'unexpected_tool_call\')\n                    text = choice[\'message\'][\'content\']\n                    log({\'usage\': response.get(\'usage\'), \'finish_reason\': choice[\'finish_reason\']})\n                    action = parse_action(text)\n                    if spec.get(\'arm\') in (\'zero\', \'entropy_pick\') and \'ask\' in action:\n                        raise ProtocolError(\'pick_required\')\n                    answer, grade = session.step(action)\n                    log({\'action\': action, \'source\': \'model\', \'answer\': answer})\n                    if grade is not None:\n                        row.update(grade)\n                        break\n                    messages.extend([{\'role\': \'assistant\', \'content\': text}, {\'role\': \'user\', \'content\': encode(answer)}])\n                else:\n                    raise ProtocolError(\'missing_pick\')\n        except (Exception,) as exc:\n            row.update(status=\'protocol_failure\' if isinstance(exc, ProtocolError) else \'infrastructure_failure\', error_type=type(exc).__name__, error=str(exc), success=False)\n        row.update(elapsed_seconds=time.monotonic() - started, questions=len(session.asked), transcript=event_path.name)\n        save_json(output / (spec[\'session_id\'] + \'.result.json\'), row)\n        records.append(row)\n        temporary = output / \'SUMMARY.tmp\'\n        temporary.write_text(json.dumps(summarize(plan, records), indent=2), encoding=\'utf-8\')\n        temporary.replace(output / \'SUMMARY.json\')\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    sub = p.add_subparsers(dest=\'command\', required=True)\n    freeze = sub.add_parser(\'freeze\')\n    freeze.add_argument(\'--controls\', action=\'store_true\', help=\'Include zero-question and entropy-assisted named arms\')\n    freeze.add_argument(\'--data\', type=Path, default=Path(__file__).with_name(\'frozen_cases.jsonl.gz\'))\n    freeze.add_argument(\'--model-info\', type=Path, required=True)\n    freeze.add_argument(\'--output\', type=Path, required=True)\n    live = sub.add_parser(\'run\')\n    live.add_argument(\'--plan\', type=Path, required=True)\n    live.add_argument(\'--data\', type=Path, default=Path(__file__).with_name(\'frozen_cases.jsonl.gz\'))\n    live.add_argument(\'--endpoint\', required=True)\n    live.add_argument(\'--tokenizer\', type=Path, required=True)\n    live.add_argument(\'--output\', type=Path, required=True)\n    verify = sub.add_parser(\'replay\')\n    verify.add_argument(\'directory\', type=Path)\n    verify.add_argument(\'--data\', type=Path, default=Path(__file__).with_name(\'frozen_cases.jsonl.gz\'))\n    report = sub.add_parser(\'summarize\')\n    report.add_argument(\'directory\', type=Path)\n    refs = sub.add_parser(\'baselines\')\n    refs.add_argument(\'--plan\', type=Path, required=True)\n    refs.add_argument(\'--data\', type=Path, default=Path(__file__).with_name(\'frozen_cases.jsonl.gz\'))\n    a = p.parse_args()\n    if a.command == \'freeze\':\n        save_json(a.output, make_plan(load_cases(a.data), sha(a.data.read_bytes()), json.loads(a.model_info.read_text(encoding=\'utf-8\')), controls=a.controls))\n    elif a.command == \'run\':\n        run(a.plan, a.data, a.endpoint, a.tokenizer, a.output)\n    elif a.command == \'replay\':\n        print(json.dumps(replay(a.directory, a.data), indent=2))\n    elif a.command == \'summarize\':\n        plan = json.loads((a.directory / \'plan.json\').read_text(encoding=\'utf-8\'))\n        rows = [json.loads(f.read_text(encoding=\'utf-8\')) for f in sorted(a.directory.glob(\'*.result.json\'))]\n        print(json.dumps(summarize(plan, rows), indent=2))\n    else:\n        plan = json.loads(a.plan.read_text(encoding=\'utf-8\'))\n        if sha(a.data.read_bytes()) != plan[\'data_sha256\']:\n            raise ValueError(\'data hash mismatch\')\n        cases = load_cases(a.data)\n        rows = []\n        for s in plan[\'schedule\']:\n            session = Session(cases[s[\'family\'], s[\'split\']], s[\'target\'], s[\'order\'], s[\'view\'], s[\'label_seed\'])\n            rows.append({\'session_id\': s[\'session_id\'], **{p: baseline(session, p) for p in [\'entropy\', \'random_split\']}})\n        print(json.dumps(rows, indent=2))\n\n\nif __name__ == \'__main__\':\n    main()\n', encoding='utf-8')


In [ ]:
reference = base64.b64decode('...')
Path("reference_frozen_cases.jsonl.gz").write_bytes(reference)
Path("frozen_cases.jsonl.gz").write_bytes(reference)


## Rebuild the reference experiment

The 18,000 rows reuse 750 target–split conditions across eight families. Scores are exact expectations for this finite resource.


In [ ]:
stream = io.StringIO()
with contextlib.redirect_stdout(stream):
    runpy.run_path("benchmark.py", run_name="__main__")
Path("summary.json").write_text(stream.getvalue(), encoding="utf-8")
print(stream.getvalue())


In [ ]:
from analyze import audit
report = audit()
Path("audit.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))


## Cross policy and probe pool

The intervention is exploratory and uses richer answer channels. The analysis computes entropy and frugal at budgets 1–3 on both scoring suites, with selected input/answer code-point costs. These are not human-time or tokenizer measurements.


In [ ]:
stream = io.StringIO()
with contextlib.redirect_stdout(stream):
    runpy.run_path("intervention.py", run_name="__main__")
Path("intervention.json").write_text(stream.getvalue(), encoding="utf-8")
print(stream.getvalue())
from paper_analysis import build
revision = build(Path("revision-results"))


## Verify, export and test the scoring boundary

Historical replay runners retain the original held-out-only endpoint. The new supervisor also checks consistency. The fixture below demonstrates why this matters; it is a deterministic software test.


In [ ]:
runpy.run_path("verify.py", run_name="__main__")
runpy.run_path("export_cases.py", run_name="__main__")
old = [json.loads(x) for x in gzip.decompress(reference).splitlines()]
new = [json.loads(x) for x in gzip.decompress(Path("frozen_cases.jsonl.gz").read_bytes()).splitlines()]
print("Frozen matrix equality under this runtime:", old == new)
# Keep the pinned matrix for the presentation control and future pilot.
Path("runtime_frozen_cases.jsonl.gz").write_bytes(Path("frozen_cases.jsonl.gz").read_bytes())
Path("frozen_cases.jsonl.gz").write_bytes(reference)
import subprocess
check = subprocess.run([sys.executable, "-B", "cognitive_probe.py", "--check"], capture_output=True, text=True, check=True)
print(check.stdout)


In [ ]:
from pilot import Session, load_cases
import random
cases = load_cases(Path("frozen_cases.jsonl.gz"))
c = cases["wrap", 43]
order = list(range(len(c["configs"])))
random.Random(2026).shuffle(order)
session = Session(c, 10, order, "named")
session.step({"ask": 9})
answer_to_model, grade = session.step({"pick": order.index(14)})
assert answer_to_model is None
assert grade["heldout_exact"] and not grade["query_consistent"] and not grade["success"]
print("Contradictory-pick fixture:", grade)
print("No model session was executed.")
